In [0]:
import requests
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import functions as F
import re

In [0]:
# =========================================
# CONFIG
# =========================================
anio_inicio = 2020
anio_actual = datetime.now().year
anio_fin = anio_actual + 5

tabla_silver = "adbsmartdatamanuelestrada.silver.tipo_cambio_bcrp_slv"

url_csv = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN01234PM/csv/{anio_inicio}-1/{anio_fin}-12/esp"

# =========================================
# 1) DESCARGAR CSV DEL BCRP
# =========================================
resp = requests.get(url_csv, timeout=60)
resp.raise_for_status()

raw = resp.text

clean = (
    raw.replace("<br>", "\n")
       .replace("&Atilde;o", "Año")
       .strip()
)

lines = [x for x in clean.split("\n") if x.strip()]

if len(lines) <= 1:
    raise ValueError("La respuesta del BCRP no contiene datos.")

data_lines = lines[1:]

# =========================================
# 2) CREAR DF BASE
# =========================================
df_lines = spark.createDataFrame([(x,) for x in data_lines], ["raw_line"])

df_src = (
    df_lines
    .withColumn("parts", F.split("raw_line", '","'))
    .withColumn("fecha_raw", F.regexp_replace(F.col("parts")[0], '^"', ''))
    .withColumn("valor_raw", F.regexp_replace(F.col("parts")[1], '"$', ''))
    .drop("parts", "raw_line")
)

# =========================================
# 3) FORMATEAR FECHA TIPO BCRP + PERIODO_TMP
# =========================================
df_src = (
    df_src
    .withColumn("mes", F.regexp_extract("fecha_raw", r"^([A-Za-z]+)", 1))
    .withColumn("anio", F.regexp_extract("fecha_raw", r"(\d{4})$", 1))
    .withColumn("fecha", F.concat(F.col("mes"), F.substring("anio", 3, 2)))
    .withColumn("valor", F.col("valor_raw").cast("double"))
    .withColumn(
        "periodo_tmp",
        F.to_date(
            F.concat_ws(
                "-",
                F.col("anio"),
                F.when(F.col("mes") == "Ene", "01")
                 .when(F.col("mes") == "Feb", "02")
                 .when(F.col("mes") == "Mar", "03")
                 .when(F.col("mes") == "Abr", "04")
                 .when(F.col("mes") == "May", "05")
                 .when(F.col("mes") == "Jun", "06")
                 .when(F.col("mes") == "Jul", "07")
                 .when(F.col("mes") == "Ago", "08")
                 .when(F.col("mes") == "Sep", "09")
                 .when(F.col("mes") == "Oct", "10")
                 .when(F.col("mes") == "Nov", "11")
                 .when(F.col("mes") == "Dic", "12"),
                F.lit("01")
            )
        )
    )
    .select("fecha", "valor", "periodo_tmp")
)

# =========================================
# 4) FORMATEAR A ESTRUCTURA GOLD (TODOS LOS PERIODOS)
# =========================================
df_gold_nuevo = (
    df_src
    .withColumnRenamed("periodo_tmp", "mes")
    .withColumn("valor", F.round(F.col("valor"), 4))
    .withColumn("data_ingestion_ts", F.current_timestamp())
    .select("mes", "valor", "data_ingestion_ts")
    .orderBy("mes")
)

total_registros = df_gold_nuevo.count()
mes_min = df_gold_nuevo.agg(F.min("mes")).collect()[0][0]
mes_max = df_gold_nuevo.agg(F.max("mes")).collect()[0][0]

print(f"\n{'='*60}")
print(f"DATOS EXTRAÍDOS DEL BCRP")
print(f"{'='*60}")
print(f"Total de registros: {total_registros}")
print(f"Mes inicial: {mes_min}")
print(f"Mes final: {mes_max}")
print(f"{'='*60}\n")

print("Vista previa (primeros 10 registros):")
display(df_gold_nuevo.limit(10))

In [0]:
# =========================================
# 6) VALIDAR SI SILVER EXISTE Y CARGAR
# =========================================
print(f"\n{'='*60}")
print("VALIDACIÓN Y CARGA A TABLA SILVER")
print(f"{'='*60}")

tabla_existe = spark.catalog.tableExists(tabla_silver)

if not tabla_existe:
    # Primera carga: crear tabla con todos los datos
    print(f"\nℹ️ La tabla {tabla_silver} NO existe. Creando...\n")
    
    df_gold_nuevo.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabla_silver)
    
    registros_insertados = df_gold_nuevo.count()
    print(f"\u2705 Tabla creada exitosamente")
    print(f"   Registros insertados: {registros_insertados}")
    print(f"   Mes inicial: {mes_min}")
    print(f"   Mes final: {mes_max}")

else:
    # Tabla existe: identificar qué periodos ya están cargados
    print(f"\nℹ️ La tabla {tabla_silver} ya existe. Validando duplicados...\n")
    
    df_existente = spark.table(tabla_silver)
    
    # Identificar registros que YA existen (solo por mes)
    df_duplicados = df_gold_nuevo.join(
        df_existente,
        df_gold_nuevo["mes"] == df_existente["mes"],
        "inner"
    ).select(df_gold_nuevo["mes"])
    
    num_duplicados = df_duplicados.count()
    
    # Filtrar solo los registros NUEVOS (no duplicados)
    df_a_insertar = df_gold_nuevo.join(
        df_duplicados,
        ["mes"],
        "left_anti"
    )
    
    num_a_insertar = df_a_insertar.count()
    
    if num_a_insertar == 0:
        print(f"\u26a0️  No hay registros nuevos para insertar")
        print(f"   Total de registros en el CSV: {total_registros}")
        print(f"   Registros ya existentes: {num_duplicados}")
        print(f"   Registros a insertar: 0")
    else:
        print(f"\u2705 Insertando registros nuevos...")
        print(f"   Total de registros en el CSV: {total_registros}")
        print(f"   Registros ya existentes: {num_duplicados}")
        print(f"   Registros a insertar: {num_a_insertar}\n")
        
        df_a_insertar.write \
            .mode("append") \
            .saveAsTable(tabla_silver)
        
        print(f"\u2705 Carga completada exitosamente")
        
        # Mostrar detalle de los meses insertados
        meses_insertados = df_a_insertar.select("mes").orderBy("mes").collect()
        if len(meses_insertados) <= 10:
            print(f"\n   Meses insertados:")
            for m in meses_insertados:
                print(f"     - {m['mes']}")
        else:
            print(f"\n   Primeros 5 meses insertados:")
            for m in meses_insertados[:5]:
                print(f"     - {m['mes']}")
            print(f"   ...(y {len(meses_insertados) - 5} más)")

print(f"\n{'='*60}")

# =========================================
# 7) PREVIEW FINAL DE LA TABLA SILVER
# =========================================
print(f"\n{'='*60}")
print("VISTA PREVIA DE LA TABLA SILVER (REGISTROS MÁS RECIENTES)")
print(f"{'='*60}\n")

df_final = spark.table(tabla_silver)
total_final = df_final.count()

print(f"Total de registros en tabla: {total_final}")
print(f"\nÚltimos 20 registros:")

display(
    df_final
    .orderBy(F.col("mes").desc())
    .limit(20)
)